# CR-1 G1 の長さの集計（CPU・API なし・数字だけ）

G1 で外れた候補が「長すぎ」か「短すぎ」かを、数字だけで集計します。問題文・応答は読みません（G1 の状態のファイルには、もともと数・ハッシュしか入っていません）。

- 費用はかかりません。GitHub のトークンも要りません。
- セル1 を実行するだけ（1 分くらい）。結果は Drive の `DECISION_POINTS/cr1/G1/CR1_G1_LENGTHS.json` に入ります。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — length summary (numbers only).
from google.colab import drive
from pathlib import Path
import json, time
drive.mount("/content/drive")

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-1 G1 長さの集計（CPU・API なし）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-1 G1 lengths', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

OUT = Path("/content/drive/MyDrive/DECISION_POINTS/cr1/G1")
S = json.loads((OUT / "CR1_G1_state.json").read_text())
T = S["templates"]
assert len(T) == 100
recs = [(int(tid), r) for tid, t in T.items() for r in t["tried"]]
fails = [r for _, r in recs if r["reason"] == "length"]
Ls = sorted(r["L"] for r in fails)
bins = {"<20": 0, "20-29": 0, "81-100": 0, "101-120": 0, "121-150": 0, "151-200": 0, ">200": 0}
for L in Ls:
    k = ("<20" if L < 20 else "20-29" if L < 30 else "81-100" if L <= 100 else "101-120" if L <= 120
         else "121-150" if L <= 150 else "151-200" if L <= 200 else ">200")
    bins[k] += 1
def pct(xs, q):
    return xs[min(len(xs) - 1, int(q * len(xs)))] if xs else None

def window_n(lo, hi):
    """Templates with a called candidate that is readable, end_turn and lo <= L <= hi (first such candidate)."""
    items = []
    for tid in sorted(T, key=int):
        for r in T[tid]["tried"]:
            if r["reason"] in (None, "length") and r["stop_reason"] == "end_turn" and r["L"] is not None and lo <= r["L"] <= hi:
                items.append({"template": int(tid), "L": r["L"], "prompt_tokens": r["prompt_tokens"], "output_tokens": r["output_tokens"]})
                break
    return items

def g1b(items, budget=90.0):
    keep = sorted(items, key=lambda it: it["template"])
    def est(xs):
        if not xs:
            return 0.0
        n = len(xs); pin = sum(x["prompt_tokens"] for x in xs) / n; pout = sum(x["output_tokens"] for x in xs) / n
        reqs = sum(x["L"] + 1 for x in xs) * 48
        return reqs * ((pin + 0.5 * pout) * 1.0 + 0.6 * pout * 5.0) / 1e6 * 0.5 * 1.2
    full = est(keep)
    while keep and est(keep) > budget:
        keep.pop()
    return {"usd_all": round(full, 2), "n_after_budget": len(keep), "usd_after_budget": round(est(keep), 2)}

windows = {}
for lo, hi in ((30, 80), (30, 100), (30, 120), (30, 150), (20, 80), (20, 120)):
    its = window_n(lo, hi)
    windows[f"{lo}-{hi}"] = {"N": len(its), "L_mean": round(sum(i["L"] for i in its) / len(its), 1) if its else None, **g1b(its)}
summary = {"schema": "cr1-g1-lengths-v1", "n_called": len(recs), "n_length_fail": len(fails),
           "too_short": sum(L < 30 for L in Ls), "too_long": sum(L > 80 for L in Ls), "length_fail_bins": bins,
           "length_fail_L_median": pct(Ls, 0.5), "length_fail_L_p90": pct(Ls, 0.9), "length_fail_L_max": Ls[-1] if Ls else None,
           "output_tokens_all_median": pct(sorted(r["output_tokens"] for _, r in recs), 0.5),
           "unreadable": sum(r["reason"] == "unreadable" for _, r in recs),
           "stop_reason_counts": {s: sum(r["stop_reason"] == s for _, r in recs) for s in {r["stop_reason"] for _, r in recs}},
           "windows": windows}
(OUT / "CR1_G1_LENGTHS.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1))
print(json.dumps(summary, ensure_ascii=False, indent=1))
notify('終わりました：長さの集計を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()